<a href="https://colab.research.google.com/github/Anshulmohan27/Machine_Learning/blob/main/Stellar_Contest_Kaggle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [112]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

In [113]:
X_train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')


In [114]:
X_train.head(
)

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [115]:
X_train.isnull().sum()/len(X_train)*100

,0
id,0.0
alpha,0.0
delta,0.0
u,0.0
g,0.0
r,0.0
i,0.0
z,0.0
redshift,0.0
spectral_type,0.0


In [116]:
test.isnull().sum()/len(test)*100

,0
id,0.0
alpha,0.0
delta,0.0
u,0.0
g,0.0
r,0.0
i,0.0
z,0.0
redshift,0.0
spectral_type,0.0


In [117]:
test_id = test['id']

def feature_engineering(df):
    df_copy = df.copy()

    # Drop 'id' column if it exists
    if 'id' in df_copy.columns:
        df_copy = df_copy.drop(columns='id')

    # Columns to apply log transformation
    columns_to_log = ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift']

    # Apply log transformation to each column, handling x=0
    for col in columns_to_log:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].apply(lambda x: np.log(x) if x != 0 else 0)

    return df_copy

# Apply feature engineering to training and test data
X_train = feature_engineering(X_train)
test = feature_engineering(test)

# Separate target variable and drop it from X_train
y_train = X_train['class']
X_train = X_train.drop(columns=['class'])

X_train.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
0,4.995415,2.830815,3.237585,3.086284,3.013470,2.957881,2.924293,-0.894083,M,Red_Sequence
1,4.851942,3.476513,3.033919,2.949011,2.867172,2.846424,2.820571,-1.845313,M,Red_Sequence
2,5.191804,3.565152,3.046197,3.048283,3.052672,3.024447,3.023219,1.038073,O/B,Blue_Cloud
3,5.419731,3.882994,3.148670,3.046936,2.945373,2.910483,2.885636,-0.623437,M,Red_Sequence
4,4.954672,2.962323,3.077458,2.968961,2.903313,2.884770,2.868818,-0.587416,M,Red_Sequence


In [118]:
X_train['spectral_type'].value_counts()

,count
spectral_type,
M,303323
A/F,122122
G/K,108546
O/B,43356


In [122]:
X_train = pd.get_dummies(X_train, columns=['spectral_type', 'galaxy_population'])
test = pd.get_dummies(test, columns=['spectral_type', 'galaxy_population'])

In [132]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
    verbose=100
)

rf.fit(X_train, y_train)
preds = rf.predict(test)



[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
building tree 1 of 150
building tree 2 of 150
building tree 3 of 150
[Parallel(n_jobs=-1)]: Done   1 tasks      | elapsed:    4.4s
building tree 4 of 150[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    5.0s

building tree 5 of 150[Parallel(n_jobs=-1)]: Done   3 tasks      | elapsed:    9.1s

building tree 6 of 150[Parallel(n_jobs=-1)]: Done   4 tasks      | elapsed:    9.6s

building tree 7 of 150[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   15.4s

building tree 8 of 150[Parallel(n_jobs=-1)]: Done   6 tasks      | elapsed:   16.3s

building tree 9 of 150
[Parallel(n_jobs=-1)]: Done   7 tasks      | elapsed:   19.6s
building tree 10 of 150[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:   20.8s

building tree 11 of 150[Parallel(n_jobs=-1)]: Done   9 tasks      | elapsed:   24.1s

building tree 12 of 150[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   26.6s

building tree 13 of 

In [133]:
print(preds)

['GALAXY' 'GALAXY' 'GALAXY' ... 'GALAXY' 'QSO' 'GALAXY']


In [134]:
preds.shape

(247435,)

In [130]:
test.shape

(247435, 14)

In [135]:
sub = pd.DataFrame(
    {
        'id': test_id,
        'class': preds
    }
)

sub.to_csv('submission.csv', index=False)